Ticker info

In [13]:
from datetime import date
import pandas as pd 
import yfinance as yf
pd.options.mode.chained_assignment = None  # default='warn'



In [16]:
BTC = yf.Ticker('BTC-USD')
info = BTC.fast_info.toJSON()
print (info)

{
    "currency": "USD",
    "dayHigh": 123850.1953125,
    "dayLow": 119392.7109375,
    "exchange": "CCC",
    "fiftyDayAverage": 113557.15078125,
    "lastPrice": 122367.234375,
    "lastVolume": 85500567552,
    "marketCap": null,
    "open": 120606.3203125,
    "previousClose": 120653.765625,
    "quoteType": "CRYPTOCURRENCY",
    "regularMarketPreviousClose": 120681.2578125,
    "shares": null,
    "tenDayAverageVolume": 54993280478,
    "threeMonthAverageVolume": 60896949215,
    "timezone": "UTC",
    "twoHundredDayAverage": 105122.311484375,
    "yearChange": 0.9443557962434281,
    "yearHigh": 124457.1171875,
    "yearLow": 58895.20703125
}


Importaçao do historicos diários . O rango maximo depende dos dados disponiveis

In [26]:
df = BTC.history(period = 'max' , interval='1h')
display(df)

,Open,High,Low,Close,Volume,Dividends,Stock Splits
Datetime,,,,,,,
2023-10-04 22:00:00+00:00,27695.802734,27773.525391,27678.767578,27756.970703,0,0.0,0.0
2023-10-04 23:00:00+00:00,27754.490234,27816.187500,27754.490234,27792.671875,0,0.0,0.0
2023-10-05 00:00:00+00:00,27798.646484,27872.707031,27732.199219,27834.103516,0,0.0,0.0
2023-10-05 01:00:00+00:00,27823.650391,27823.650391,27699.457031,27737.802734,0,0.0,0.0
2023-10-05 02:00:00+00:00,27737.802734,27739.777344,27675.664062,27677.197266,0,0.0,0.0
...,...,...,...,...,...,...,...
2025-10-03 18:00:00+00:00,122137.765625,122749.335938,121576.054688,122480.968750,2452504576,0.0,0.0
2025-10-03 19:00:00+00:00,122598.406250,122973.546875,122460.437500,122724.007812,11690156032,0.0,0.0
2025-10-03 20:00:00+00:00,122736.218750,122794.953125,122130.867188,122598.007812,6003015680,0.0,0.0


In [28]:
import sqlite3
import pandas as pd

# Caminho do banco
db_path = r'C:\Users\scitr\anaconda_projects\Trading_System\Dados_Fontes\sqtitulos.db'
conn = sqlite3.connect(db_path)

# ID fixo
id_titulosintervalos = 3

# Copiar o DataFrame e remover o sufixo +00:00 do índice
df = df.copy()
df['datetime'] = df.index.strftime('%Y-%m-%d %H:%M:%S')  # Remove o +00:00

# Buscar datetimes já existentes no banco para esse intervalo
query = """
SELECT datetime FROM tbtitulosprezos
WHERE id_titulosintervalos = ?
"""
existing = pd.read_sql(query, conn, params=(id_titulosintervalos,))
existing_set = set(existing['datetime'])

# Filtrar apenas os novos registros
df_filtered = df[~df['datetime'].isin(existing_set)].copy()

# Adicionar coluna de id_titulosintervalos
df_filtered['id_titulosintervalos'] = id_titulosintervalos

# Reorganizar colunas conforme a estrutura da tabela
df_filtered = df_filtered[['id_titulosintervalos', 'datetime', 'Open', 'High', 'Low', 'Close', 'Volume']]
df_filtered.columns = ['id_titulosintervalos', 'datetime', 'open', 'high', 'low', 'close', 'volume']

# Inserir no banco
df_filtered.to_sql('tbtitulosprezos', conn, if_exists='append', index=False)

# Fechar conexão
conn.close()

Extrae dividendos e splits

In [32]:
df_filtrado = df[(df['Dividends'] != 0) | (df['Stock Splits'] != 0)]
display(df_filtrado)

,Open,High,Low,Close,Volume,Dividends,Stock Splits
Datetime,,,,,,,
